In [1]:
import json
import pandas as pd

RAW_FILE = "../dataset/raw_jobs.json"

with open(RAW_FILE, "r", encoding="utf-8") as file:
    raw_data = json.load(file)

print("Source:", raw_data["source"])
print("Collection Time:", raw_data["collected_at_utc"])
print("Total Pages:", raw_data["last_completed_page"])
print("Total Jobs:", raw_data["total_jobs"])

Source: https://www.arbeitnow.com/api/job-board-api
Collection Time: 2026-10-01T06:55:29.462595+00:00
Total Pages: 26
Total Jobs: 2995


In [2]:
jobs = raw_data["data"]

df = pd.DataFrame(jobs)

print("Dataset Shape:", df.shape)

df.head()

Dataset Shape: (2995, 10)


,slug,company_name,title,description,remote,url,tags,job_types,location,created_at
0,software-engineer-price-comparison-portal-berl...,Preiswecker,Software Engineer,<p>Preiswecker is a price comparison portal in...,False,https://www.preiswecker.com/,"[Software Engineering, E-Commerce]",[Full-time],Berlin,1786516800
1,senior-product-manager-356253,Verve,Senior Product Manager,&lt;div class=&quot;content-intro&quot;&gt;&lt...,False,https://www.arbeitnow.com/jobs/companies/verve...,[Product Management],[],,1790834115
2,working-student-launch-operations-parsdorf-bav...,isaraerospace,Working Student Launch Operations (m/f/d),&lt;p&gt;&lt;strong&gt;&lt;span lang=&quot;EN-...,False,https://www.arbeitnow.com/jobs/companies/isara...,[Mission & Launch Operations],[],"Parsdorf, Bavaria",1790834111
3,senior-mechanical-engineer-parsdorf-bavaria-38...,isaraerospace,Senior Mechanical Engineer (f/m/d),&lt;p class=&quot;p1&quot;&gt;&lt;strong&gt;Mi...,False,https://www.arbeitnow.com/jobs/companies/isara...,[Ground Systems],[],"Parsdorf, Bavaria",1790834111
4,mes-process-and-system-engineer-parsdorf-bavar...,isaraerospace,MES Process and System Engineer (m/f/d),&lt;p&gt;&lt;strong&gt;Mission Brief&amp;nbsp;...,False,https://www.arbeitnow.com/jobs/companies/isara...,"[Production, Operations & Quality]",[],"Parsdorf, Bavaria",1790834111


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2995 entries, 0 to 2994
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   slug          2995 non-null   str   
 1   company_name  2995 non-null   str   
 2   title         2995 non-null   str   
 3   description   2995 non-null   str   
 4   remote        2995 non-null   bool  
 5   url           2995 non-null   str   
 6   tags          2995 non-null   object
 7   job_types     2995 non-null   object
 8   location      2995 non-null   str   
 9   created_at    2995 non-null   int64 
dtypes: bool(1), int64(1), object(2), str(6)
memory usage: 213.6+ KB


In [4]:
print(df.columns.tolist())

['slug', 'company_name', 'title', 'description', 'remote', 'url', 'tags', 'job_types', 'location', 'created_at']


In [5]:
df.shape

(2995, 10)

In [7]:
print("Unique companies:", df["company_name"].nunique())
print("Unique locations:", df["location"].nunique())
print("Unique job titles:", df["title"].nunique())

Unique companies: 1271
Unique locations: 634
Unique job titles: 2583


In [10]:
# Create a temporary copy for duplicate checking
df_duplicate_check = df.copy()

# Convert list columns to tuples
df_duplicate_check["tags"] = df_duplicate_check["tags"].apply(tuple)
df_duplicate_check["job_types"] = df_duplicate_check["job_types"].apply(tuple)

# Check duplicates
print("Duplicate rows:", df_duplicate_check.duplicated().sum())
print("Duplicate slugs:", df["slug"].duplicated().sum())
print("Duplicate URLs:", df["url"].duplicated().sum())

Duplicate rows: 63
Duplicate slugs: 122
Duplicate URLs: 122


In [11]:
print("Remote job distribution:")
print(df["remote"].value_counts())

print("\nRemote percentage:")
print(
    df["remote"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

Remote job distribution:
remote
False    2777
True      218
Name: count, dtype: int64

Remote percentage:
remote
False    92.72
True      7.28
Name: proportion, dtype: float64


In [12]:
df["created_at_date"] = pd.to_datetime(
    df["created_at"],
    unit="s",
    utc=True
)

print("Earliest posting:")
print(df["created_at_date"].min())

print("\nLatest posting:")
print(df["created_at_date"].max())

Earliest posting:
2026-08-12 06:40:00+00:00

Latest posting:
2026-10-01 05:55:15+00:00


In [13]:
job_type_available = df["job_types"].apply(
    lambda x: len(x) > 0
)

print("Jobs with job type:", job_type_available.sum())
print("Jobs without job type:", (~job_type_available).sum())

print(
    "Job type coverage:",
    round(job_type_available.mean() * 100, 2),
    "%"
)

Jobs with job type: 2094
Jobs without job type: 901
Job type coverage: 69.92 %


In [14]:
tag_available = df["tags"].apply(
    lambda x: len(x) > 0
)

print("Jobs with tags:", tag_available.sum())
print("Jobs without tags:", (~tag_available).sum())

print(
    "Tag coverage:",
    round(tag_available.mean() * 100, 2),
    "%"
)

Jobs with tags: 2739
Jobs without tags: 256
Tag coverage: 91.45 %
